# ARMADO DEL DATASET POST API

# DATOS

“El dataset base proviene de Make Music Equal (licencia CC BY-SA 4.0).
Este fue enriquecido con datos obtenidos mediante la API de Chartmetric, los cuales poseen restricciones de uso.
Por este motivo, las variables derivadas de dicha fuente fueron transformadas y agregadas de forma tal que no permiten reconstruir los valores originales.”

PASO 1: Base de artistas para obtener IDs antes de conectar a la API
Fuente: Make Music Equal (CC BY-SA 4.0)
La base de datos de artistas se descargó del dataset de acceso público MakeMusicEqual versión 3/3/2026: 936420, 10.
https://chartmetric-public.s3.us-west-2.amazonaws.com/make-music-equal/mme_artist_info.csv 
Se eliminaron registros duplicados [nombre + url de artista]: 172
Resultado: 936248 filas/artistas. 
Legal: This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.


PASO 2: Construcción de las cohortes de datos a extraer
Base de datos MakeMusicEqual ordenada por Chartmetric_rank (rank 1 a 936248)
-cohorte 1: 1400 registros 
-cohorte 2: 600 registros 

Notas:
Hay discontinuidades en el atributo Chartmetric_rank (faltan números en la secuencia) porque la base makemusicequal está actualizada al 3/3/26 aunque fueron descargados el 23/3/2026. Misma fecha de acceso a la API. Como la variable chartmetrick_rank es dinámica, esa diferencia de 20 días hace que un artista pueda tener rank = 5 el 3/3/2026 y rank = 9 el 23/3/26. Sin embargo, eso no invalida la selección del conjunto de artistas sobre la cual se construye la extracción de datos porque el objetivo fue definir un conjunto de artistas ordenados por popularidad (medida por una fuenta de datos confiable). Además la variable rank no será utilizada en el análisis. Y por último, se comprueba que el rank puede variar sutilmente pero el conjunto de artistas definidos es predominantemente coincidente. 

PASO 3: API. Extracción de datos
Límite 14 días, 100 request x día. (inicio 23/3/26)
Se apunta a dos endpoints: artist metadaga y artist live events
La extracción de datos se fragmenta por cohortes.


desdargas cohorte 1: 23 a 25 de marzo (ajustando fecha de live events)
el listado de artistas lo saco de mmequal ordenado por rank eso es del (6 de marzo)


In [3]:
import pandas as pd
import numpy as np

# live events agregado

In [4]:
ruta_eventlevel = r"chartmetric_liveevents_2024_2025_cohorte2\liveevents_2024_2025_eventlevel_cohorte2.csv"

In [5]:
df_events = pd.read_csv(ruta_eventlevel)

In [6]:
print(df_events.shape)
print(df_events.columns.tolist())
df_events.head(3)

(70057, 24)
['downloaded_at_utc', 'chartmetric_id', 'artist_name', 'event_id', 'event_name', 'type', 'event_year', 'start_date', 'end_date', 'venue_id', 'venue_name', 'venue_capacity', 'city_id', 'city_name', 'code2', 'price', 'low_price', 'high_price', 'currency', 'is_headliner', 'jambase_event_url', 'songkick_event_url', 'seatgeek_event_url', 'ticketmaster_event_url']


,downloaded_at_utc,chartmetric_id,artist_name,event_id,event_name,type,event_year,start_date,end_date,venue_id,...,code2,price,low_price,high_price,currency,is_headliner,jambase_event_url,songkick_event_url,seatgeek_event_url,ticketmaster_event_url
0,2026-03-25T15:51:37.237261+00:00,1103173,Murilo Huff,9812380,Murilo Huff at Na Praia Parque de Experiências,Concert,2025,2025-08-09 18:00:00.000,2025-08-09 23:59:59.000,NaN,...,NaN,NaN,NaN,NaN,NaN,True,https://www.jambase.com/show/murilo-huff-na-pr...,NaN,NaN,NaN
1,2026-03-25T15:51:37.237272+00:00,1103173,Murilo Huff,8498484,Murilo Huff at Electric Brixton,Concert,2025,2025-02-20 19:00:00.000,2025-02-20 23:59:59.000,2194.0,...,GB,NaN,NaN,NaN,NaN,True,https://www.jambase.com/show/murilo-huff-elect...,NaN,NaN,http://ticketmaster-uk.tm7559.net/c/252938/431...
2,2026-03-25T15:51:37.237278+00:00,1103173,Murilo Huff,169672,Marília Mendonça at Allianz Parque,Concert,2024,2024-10-05 15:00:00.000,2024-10-05 23:59:59.000,7086.0,...,BR,NaN,NaN,NaN,NaN,True,https://www.jambase.com/show/marilia-mendonca-...,https://www.songkick.com/festivals/3688427-thi...,NaN,NaN


In [7]:
# fechas
df_events["start_date"] = pd.to_datetime(df_events["start_date"], errors="coerce")
df_events["end_date"] = pd.to_datetime(df_events["end_date"], errors="coerce")

# por seguridad, si event_year vino mal tipado
df_events["event_year"] = pd.to_numeric(df_events["event_year"], errors="coerce")

# columnas numéricas que vamos a usar
cols_numericas = ["low_price", "high_price", "venue_capacity"]

for col in cols_numericas:
    df_events[col] = pd.to_numeric(df_events[col], errors="coerce")

In [8]:
print(df_events["event_year"].value_counts(dropna=False).sort_index())

event_year
2024    25825
2025    44231
2026        1
Name: count, dtype: int64


In [9]:
def moneda_principal(serie):
    serie_limpia = serie.dropna()
    if serie_limpia.empty:
        return np.nan
    return serie_limpia.mode().iloc[0]

In [10]:
def resumir_live_por_anio(df, anio):
    df_anio = df[df["event_year"] == anio].copy()
    
    resumen = (
        df_anio
        .groupby(["chartmetric_id", "artist_name"], as_index=False)
        .agg(
            n_shows=("event_id", "nunique"),
            avg_low_price=("low_price", "mean"),
            avg_high_price=("high_price", "mean"),
            avg_venue_capacity=("venue_capacity", "mean"),
            total_capacity=("venue_capacity", "sum"),
            n_shows_with_capacity=("venue_capacity", lambda x: x.notna().sum()),
            n_cities=("city_name", lambda x: x.dropna().nunique()),
            n_countries=("code2", lambda x: x.dropna().nunique()),
            n_currencies=("currency", lambda x: x.dropna().nunique()),
            main_currency=("currency", moneda_principal)
        )
    )
    
    resumen = resumen.rename(columns={
        "n_shows": f"n_shows_{anio}",
        "avg_low_price": f"avg_low_price_{anio}",
        "avg_high_price": f"avg_high_price_{anio}",
        "avg_venue_capacity": f"avg_venue_capacity_{anio}",
        "total_capacity": f"total_capacity_{anio}",
        "n_shows_with_capacity": f"n_shows_with_capacity_{anio}",
        "n_cities": f"n_cities_{anio}",
        "n_countries": f"n_countries_{anio}",
        "n_currencies": f"n_currencies_{anio}",
        "main_currency": f"main_currency_{anio}"
    })
    
    # si no hay ningún dato de capacidad, dejar total_capacity como NaN
    col_total = f"total_capacity_{anio}"
    col_ncap = f"n_shows_with_capacity_{anio}"
    resumen.loc[resumen[col_ncap] == 0, col_total] = np.nan
    
    return resumen

In [11]:
df_live_2024 = resumir_live_por_anio(df_events, 2024)
df_live_2025 = resumir_live_por_anio(df_events, 2025)

print("df_live_2024:", df_live_2024.shape)
print("df_live_2025:", df_live_2025.shape)

df_live_2024: (1982, 12)
df_live_2025: (2259, 12)


In [12]:
df_live_agg = df_live_2024.merge(
    df_live_2025,
    on=["chartmetric_id", "artist_name"],
    how="outer"
)

In [13]:
cols_cero = [
    "n_shows_2024",
    "n_cities_2024",
    "n_countries_2024",
    "n_currencies_2024",
    "n_shows_with_capacity_2024",
    "n_shows_2025",
    "n_cities_2025",
    "n_countries_2025",
    "n_currencies_2025",
    "n_shows_with_capacity_2025"
]

for col in cols_cero:
    if col in df_live_agg.columns:
        df_live_agg[col] = df_live_agg[col].fillna(0)

In [14]:
df_live_agg["shows_per_country_2025"] = np.where(
    df_live_agg["n_countries_2025"] > 0,
    df_live_agg["n_shows_2025"] / df_live_agg["n_countries_2025"],
    np.nan
)

df_live_agg["shows_per_country_2024"] = np.where(
    df_live_agg["n_countries_2024"] > 0,
    df_live_agg["n_shows_2024"] / df_live_agg["n_countries_2024"],
    np.nan
)

## columnas bianuales

In [15]:
# columnas combinadas 2024 + 2025
df_live_agg["n_shows_total"] = (
    df_live_agg["n_shows_2024"] + df_live_agg["n_shows_2025"]
)

df_live_agg["n_shows_with_capacity_total"] = (
    df_live_agg["n_shows_with_capacity_2024"] + df_live_agg["n_shows_with_capacity_2025"]
)

df_live_agg["total_capacity_total"] = (
    df_live_agg["total_capacity_2024"].fillna(0) + df_live_agg["total_capacity_2025"].fillna(0)
)

# si no hubo ningún show con capacidad, dejar total_capacity_total como NaN
df_live_agg.loc[
    df_live_agg["n_shows_with_capacity_total"] == 0,
    "total_capacity_total"
] = np.nan

df_live_agg["avg_venue_capacity_total"] = np.where(
    df_live_agg["n_shows_with_capacity_total"] > 0,
    df_live_agg["total_capacity_total"] / df_live_agg["n_shows_with_capacity_total"],
    np.nan
)



In [16]:
def resumir_live_total(df):
    df_total = df[df["event_year"].isin([2024, 2025])].copy()
    
    resumen_total = (
        df_total
        .groupby(["chartmetric_id", "artist_name"], as_index=False)
        .agg(
            n_cities_total=("city_name", lambda x: x.dropna().nunique()),
            n_countries_total=("code2", lambda x: x.dropna().nunique())
        )
    )
    
    return resumen_total

In [17]:
df_live_total = resumir_live_total(df_events)

df_live_agg = df_live_agg.merge(
    df_live_total,
    on=["chartmetric_id", "artist_name"],
    how="left"
)

In [18]:
df_live_agg["shows_per_country_total"] = np.where(
    df_live_agg["n_countries_total"] > 0,
    df_live_agg["n_shows_total"] / df_live_agg["n_countries_total"],
    np.nan
)

In [19]:
print(df_live_agg.shape)
print(df_live_agg.columns.tolist())
df_live_agg.head(10)

(2512, 31)
['chartmetric_id', 'artist_name', 'n_shows_2024', 'avg_low_price_2024', 'avg_high_price_2024', 'avg_venue_capacity_2024', 'total_capacity_2024', 'n_shows_with_capacity_2024', 'n_cities_2024', 'n_countries_2024', 'n_currencies_2024', 'main_currency_2024', 'n_shows_2025', 'avg_low_price_2025', 'avg_high_price_2025', 'avg_venue_capacity_2025', 'total_capacity_2025', 'n_shows_with_capacity_2025', 'n_cities_2025', 'n_countries_2025', 'n_currencies_2025', 'main_currency_2025', 'shows_per_country_2025', 'shows_per_country_2024', 'n_shows_total', 'n_shows_with_capacity_total', 'total_capacity_total', 'avg_venue_capacity_total', 'n_cities_total', 'n_countries_total', 'shows_per_country_total']


,chartmetric_id,artist_name,n_shows_2024,avg_low_price_2024,avg_high_price_2024,avg_venue_capacity_2024,total_capacity_2024,n_shows_with_capacity_2024,n_cities_2024,n_countries_2024,...,main_currency_2025,shows_per_country_2025,shows_per_country_2024,n_shows_total,n_shows_with_capacity_total,total_capacity_total,avg_venue_capacity_total,n_cities_total,n_countries_total,shows_per_country_total
0,5,10cc,55.0,97.941176,326.470588,1625.555556,73150.0,45.0,44.0,6.0,...,EUR,7.600000,9.166667,93.0,74.0,130568.0,1764.432432,77,9,10.333333
1,7,Bush,41.0,53.814286,6711.812857,9465.078947,359673.0,38.0,35.0,2.0,...,USD,4.666667,20.500000,125.0,114.0,1392720.0,12216.842105,101,18,6.944444
2,13,Cat Power,63.0,92.702632,6475.118421,5054.759259,272957.0,54.0,54.0,12.0,...,EUR,2.076923,5.250000,90.0,70.0,311951.0,4456.442857,75,19,4.736842
3,14,Garbage,13.0,NaN,NaN,3618.545455,39804.0,11.0,12.0,7.0,...,USD,5.444444,1.857143,62.0,53.0,229770.0,4335.283019,59,16,3.875000
4,20,Modest Mouse,50.0,79.954545,10967.045455,5583.571429,273595.0,49.0,35.0,2.0,...,USD,30.500000,25.000000,111.0,110.0,511442.0,4649.472727,74,2,55.500000
5,24,Cypress Hill,6.0,NaN,NaN,6036.000000,12072.0,2.0,5.0,4.0,...,USD,7.166667,1.500000,49.0,39.0,240014.0,6154.205128,44,8,6.125000
6,25,Madness,20.0,NaN,NaN,8433.333333,75900.0,9.0,19.0,5.0,...,EUR,4.500000,4.000000,47.0,32.0,327311.0,10228.468750,39,8,5.875000
7,36,New Edition,9.0,268.571429,3004.000000,1500.000000,13500.0,9.0,1.0,1.0,...,USD,7.000000,9.000000,16.0,16.0,25500.0,1593.750000,2,1,16.000000
8,38,James Taylor,21.0,61.794118,2629.382353,15502.842105,294554.0,19.0,13.0,1.0,...,USD,22.500000,21.000000,66.0,64.0,789792.0,12340.500000,44,2,33.000000
9,39,Styx,47.0,42.875000,658.250000,12425.152174,571557.0,46.0,42.0,1.0,...,USD,41.000000,47.000000,129.0,123.0,1381650.0,11232.926829,89,2,64.500000


In [21]:
print("Duplicados por chartmetric_id:", df_live_agg["chartmetric_id"].duplicated().sum())
print("Duplicados por artist_name:", df_live_agg["artist_name"].duplicated().sum())

Duplicados por chartmetric_id: 0
Duplicados por artist_name: 0


In [22]:
df_live_agg.isna().sum().sort_values(ascending=False)

avg_low_price_2024             1339
avg_high_price_2024            1339
main_currency_2024             1159
avg_low_price_2025             1053
avg_high_price_2025            1053
main_currency_2025              923
total_capacity_2024             671
avg_venue_capacity_2024         671
shows_per_country_2024          552
avg_venue_capacity_2025         391
total_capacity_2025             391
shows_per_country_2025          290
total_capacity_total            123
avg_venue_capacity_total        123
shows_per_country_total          24
n_shows_2025                      0
n_cities_2024                     0
n_countries_2024                  0
n_currencies_2024                 0
n_shows_with_capacity_2024        0
n_shows_2024                      0
chartmetric_id                    0
artist_name                       0
n_countries_2025                  0
n_currencies_2025                 0
n_shows_with_capacity_2025        0
n_cities_2025                     0
n_shows_with_capacity_total 

# SAVE df_live_agg

In [23]:
ruta_salida_live_agg = r"chartmetric_liveevents_2024_2025_cohorte2\liveevents_2024_2025_artist_agg_cohorte2.csv"
df_live_agg.to_csv(ruta_salida_live_agg, index=False)

In [24]:
df_live_agg.shape


(2512, 31)

In [25]:
df_live_agg.columns


Index(['chartmetric_id', 'artist_name', 'n_shows_2024', 'avg_low_price_2024',
       'avg_high_price_2024', 'avg_venue_capacity_2024', 'total_capacity_2024',
       'n_shows_with_capacity_2024', 'n_cities_2024', 'n_countries_2024',
       'n_currencies_2024', 'main_currency_2024', 'n_shows_2025',
       'avg_low_price_2025', 'avg_high_price_2025', 'avg_venue_capacity_2025',
       'total_capacity_2025', 'n_shows_with_capacity_2025', 'n_cities_2025',
       'n_countries_2025', 'n_currencies_2025', 'main_currency_2025',
       'shows_per_country_2025', 'shows_per_country_2024', 'n_shows_total',
       'n_shows_with_capacity_total', 'total_capacity_total',
       'avg_venue_capacity_total', 'n_cities_total', 'n_countries_total',
       'shows_per_country_total'],
      dtype='str')

In [26]:
df_live_agg.head()


,chartmetric_id,artist_name,n_shows_2024,avg_low_price_2024,avg_high_price_2024,avg_venue_capacity_2024,total_capacity_2024,n_shows_with_capacity_2024,n_cities_2024,n_countries_2024,...,main_currency_2025,shows_per_country_2025,shows_per_country_2024,n_shows_total,n_shows_with_capacity_total,total_capacity_total,avg_venue_capacity_total,n_cities_total,n_countries_total,shows_per_country_total
0,5,10cc,55.0,97.941176,326.470588,1625.555556,73150.0,45.0,44.0,6.0,...,EUR,7.600000,9.166667,93.0,74.0,130568.0,1764.432432,77,9,10.333333
1,7,Bush,41.0,53.814286,6711.812857,9465.078947,359673.0,38.0,35.0,2.0,...,USD,4.666667,20.500000,125.0,114.0,1392720.0,12216.842105,101,18,6.944444
2,13,Cat Power,63.0,92.702632,6475.118421,5054.759259,272957.0,54.0,54.0,12.0,...,EUR,2.076923,5.250000,90.0,70.0,311951.0,4456.442857,75,19,4.736842
3,14,Garbage,13.0,NaN,NaN,3618.545455,39804.0,11.0,12.0,7.0,...,USD,5.444444,1.857143,62.0,53.0,229770.0,4335.283019,59,16,3.875000
4,20,Modest Mouse,50.0,79.954545,10967.045455,5583.571429,273595.0,49.0,35.0,2.0,...,USD,30.500000,25.000000,111.0,110.0,511442.0,4649.472727,74,2,55.500000


In [27]:
df_live_agg.isna().sum().sort_values(ascending=False).head(15)

avg_low_price_2024          1339
avg_high_price_2024         1339
main_currency_2024          1159
avg_low_price_2025          1053
avg_high_price_2025         1053
main_currency_2025           923
total_capacity_2024          671
avg_venue_capacity_2024      671
shows_per_country_2024       552
avg_venue_capacity_2025      391
total_capacity_2025          391
shows_per_country_2025       290
total_capacity_total         123
avg_venue_capacity_total     123
shows_per_country_total       24
dtype: int64

In [28]:
print("Artistas únicos en eventlevel:", df_events["chartmetric_id"].nunique())
print("Filas en df_live_agg:", df_live_agg.shape[0])

Artistas únicos en eventlevel: 2512
Filas en df_live_agg: 2512


In [29]:
df_live_summary = pd.read_csv(
    r"chartmetric_liveevents_2024_2025_cohorte2\liveevents_2024_2025_summary_cohorte2.csv"
)

In [30]:
print("Artistas en live summary:", df_live_summary["chartmetric_id"].nunique())

Artistas en live summary: 4482


## all artists vs artists con shows

In [33]:
ids_eventlevel = set(df_live_agg["chartmetric_id"].unique())
ids_summary = set(df_live_summary["chartmetric_id"].unique())

faltan_en_eventlevel = ids_summary - ids_eventlevel

print("Artistas en live_summary:", len(ids_summary))
print("Artistas en df_live_agg:", len(ids_eventlevel))
print("Artistas que están en summary pero no en eventlevel:", len(faltan_en_eventlevel))

Artistas en live_summary: 4482
Artistas en df_live_agg: 2512
Artistas que están en summary pero no en eventlevel: 1970


In [34]:
df_faltantes = df_live_summary[
    df_live_summary["chartmetric_id"].isin(faltan_en_eventlevel)
].copy()

print(df_faltantes.shape)
df_faltantes.head(10)

(1970, 13)


,downloaded_at_utc,chartmetric_id,artist_name,ok,status_code,error_text,raw_json_path,n_shows_2024,n_shows_2025,n_requests_live_2024_2025,x_ratelimit_limit,x_ratelimit_remaining,x_ratelimit_reset
2,2026-03-25T15:52:34.460015+00:00,113,Whitesnake,True,200,NaN,chartmetric_liveevents_2024_2025_cohorte2\raw_...,0,0,1,1000,55,1774515389
13,2026-03-25T15:53:58.830785+00:00,1423874,Guilherme & Benuto,True,200,NaN,chartmetric_liveevents_2024_2025_cohorte2\raw_...,0,0,1,1000,44,1774515389
16,2026-03-25T15:54:08.063404+00:00,50,Savage Garden,True,200,NaN,chartmetric_liveevents_2024_2025_cohorte2\raw_...,0,0,1,1000,41,1774515389
17,2026-03-25T15:54:16.977920+00:00,168422,John Martin,True,200,NaN,chartmetric_liveevents_2024_2025_cohorte2\raw_...,0,0,1,1000,40,1774515389
18,2026-03-25T15:54:21.674917+00:00,434307,Joe Jonas,True,200,NaN,chartmetric_liveevents_2024_2025_cohorte2\raw_...,0,0,1,1000,39,1774515389
19,2026-03-25T15:54:24.077272+00:00,814,The Hollies,True,200,NaN,chartmetric_liveevents_2024_2025_cohorte2\raw_...,0,0,1,1000,38,1774515389
21,2026-03-25T15:54:28.505406+00:00,18975,D.O.D,True,200,NaN,chartmetric_liveevents_2024_2025_cohorte2\raw_...,0,0,1,1000,36,1774515389
24,2026-03-25T15:54:38.229789+00:00,29975,Bomfunk MC's,True,200,NaN,chartmetric_liveevents_2024_2025_cohorte2\raw_...,0,0,1,1000,33,1774515389
27,2026-03-25T15:54:45.958894+00:00,1714066,Majed,True,200,NaN,chartmetric_liveevents_2024_2025_cohorte2\raw_...,0,0,1,1000,30,1774515389
28,2026-03-25T15:54:48.210021+00:00,145086,Alex Rose,True,200,NaN,chartmetric_liveevents_2024_2025_cohorte2\raw_...,0,0,1,1000,29,1774515389


In [35]:
print(df_faltantes[["ok", "status_code"]].value_counts(dropna=False))

ok    status_code
True  200            1970
Name: count, dtype: int64


In [36]:
print("Distribución n_shows_2024 en faltantes:")
print(df_faltantes["n_shows_2024"].value_counts(dropna=False).sort_index())

print("\nDistribución n_shows_2025 en faltantes:")
print(df_faltantes["n_shows_2025"].value_counts(dropna=False).sort_index())

Distribución n_shows_2024 en faltantes:
n_shows_2024
0    1970
Name: count, dtype: int64

Distribución n_shows_2025 en faltantes:
n_shows_2025
0    1970
Name: count, dtype: int64


In [37]:
condicion_esperada = (
    (df_faltantes["ok"] == True) &
    (df_faltantes["status_code"] == 200) &
    (df_faltantes["n_shows_2024"] == 0) &
    (df_faltantes["n_shows_2025"] == 0)
)

print("Cantidad de faltantes que cumplen exactamente la condición esperada:", condicion_esperada.sum())
print("Total de faltantes:", len(df_faltantes))

Cantidad de faltantes que cumplen exactamente la condición esperada: 1970
Total de faltantes: 1970


In [38]:
df_casos_raros = df_faltantes[~condicion_esperada].copy()

print(df_casos_raros.shape)
df_casos_raros.head(20)

(0, 13)


,downloaded_at_utc,chartmetric_id,artist_name,ok,status_code,error_text,raw_json_path,n_shows_2024,n_shows_2025,n_requests_live_2024_2025,x_ratelimit_limit,x_ratelimit_remaining,x_ratelimit_reset


## AGREGO BAND al flat.csv

In [41]:
import os
import json
import pandas as pd

JSON_DIR = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\chartmetric_metadata_cohorte2\raw_json_cohorte2"
CSV_PATH = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\chartmetric_metadata_cohorte2\metadata_flat_cohorte2.csv"


# ------------------------------------------
# 1) extraer id + band desde JSON
# ------------------------------------------
rows_band = []

json_files = [f for f in os.listdir(JSON_DIR) if f.lower().endswith(".json")]

for fname in json_files:
    fpath = os.path.join(JSON_DIR, fname)

    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)

        obj = data.get("obj", data)

        rows_band.append({
            "response_id": obj.get("id"),   # ← clave alineada con tu CSV
            "band": obj.get("band")
        })

    except Exception as e:
        print(f"Error leyendo {fname}: {e}")

df_band = pd.DataFrame(rows_band).drop_duplicates(subset=["response_id"])

# ------------------------------------------
# 2) leer metadata_flat
# ------------------------------------------
df_metadata = pd.read_csv(CSV_PATH)

# ------------------------------------------
# 3) merge limpio (sin tocar nada más)
# ------------------------------------------
df_metadata = df_metadata.drop(columns=["band"], errors="ignore")

df_metadata = df_metadata.merge(
    df_band,
    on="response_id",
    how="left"
)

# ------------------------------------------
# 4) guardar
# ------------------------------------------
df_metadata.to_csv(CSV_PATH, index=False)

print("metadata_flat.csv actualizado con la columna band")
print(df_metadata["band"].value_counts(dropna=False))

metadata_flat.csv actualizado con la columna band
band
False    3174
True     1308
Name: count, dtype: int64


# MERGE metadata

In [42]:
ruta_metadata = r"chartmetric_metadata_cohorte2\metadata_flat_cohorte2.csv"

df_metadata = pd.read_csv(ruta_metadata)

print(df_metadata.shape)
df_metadata.head()

(4482, 34)


,downloaded_at_utc,requested_chartmetric_id,requested_artist_name,response_id,name,code2,pronoun_title,gender_title,record_label,hometown_city,...,tiktok_likes,ycs_subscribers,ycs_views,youtube_daily_video_views,youtube_monthly_video_views,deezer_fans,shazam_count,pandora_lifetime_streams,pandora_lifetime_stations_added,band
0,2026-03-25T15:39:41.407622,1103173,Murilo Huff,1103173,Murilo Huff,BR,he/him,male,M Show,Goiania,...,35700000.0,2050000.0,2.849435e+09,NaN,NaN,1264953.0,1123354.0,7.640000e+02,1.0,False
1,2026-03-25T15:39:43.603430,776,Queens of the Stone Age,776,Queens of the Stone Age,US,they/them,male,Matador,NaN,...,514800.0,923000.0,1.015514e+09,352942.0,10151743.0,1349033.0,11006773.0,3.253573e+08,1654604.0,True
2,2026-03-25T15:40:53.890832,113,Whitesnake,113,Whitesnake,GB,they/them,NaN,WMG,London,...,NaN,510000.0,4.699978e+08,464137.0,13973038.0,616038.0,23302259.0,1.253825e+09,1472479.0,True
3,2026-03-25T15:41:00.428819,7622,Espinoza Paz,7622,Espinoza Paz,MX,he/him,male,Espinoza Paz,Mazatlán,...,21800000.0,3800000.0,2.670555e+09,1700109.0,38216824.0,15.0,8780432.0,6.327822e+08,5217747.0,False
4,2026-03-25T15:41:06.287624,566366,La Joaqui,566366,La Joaqui,AR,she/her,NaN,LA JOAQUI / DALE PLAY Records,NaN,...,169700000.0,1530000.0,3.897122e+07,1141022.0,31848728.0,16579.0,2885366.0,6.567140e+05,3543.0,False


In [43]:
df_metadata = df_metadata.rename(columns={
    "response_id": "chartmetric_id"
})

In [44]:
print("Duplicados en metadata:", df_metadata["chartmetric_id"].duplicated().sum())

Duplicados en metadata: 0


In [45]:
print(df_metadata.columns.tolist())

['downloaded_at_utc', 'requested_chartmetric_id', 'requested_artist_name', 'chartmetric_id', 'name', 'code2', 'pronoun_title', 'gender_title', 'record_label', 'hometown_city', 'cm_artist_rank', 'cm_artist_score', 'career_stage', 'career_stage_score', 'career_trend', 'career_trend_score', 'primary_genre', 'sp_followers', 'sp_monthly_listeners', 'sp_popularity', 'sp_playlist_total_reach', 'ins_followers', 'twitter_followers', 'tiktok_followers', 'tiktok_likes', 'ycs_subscribers', 'ycs_views', 'youtube_daily_video_views', 'youtube_monthly_video_views', 'deezer_fans', 'shazam_count', 'pandora_lifetime_streams', 'pandora_lifetime_stations_added', 'band']


In [46]:
df_metadata = df_metadata.rename(columns={
    "name": "artist_name"
})

# MERGE CRITICO

In [47]:
df_final = df_metadata.merge(
    df_live_agg,
    on="chartmetric_id",
    how="left",
    suffixes=("_meta", "_live")
)

In [48]:
[col for col in df_final.columns if "name" in col.lower()]

['requested_artist_name', 'artist_name_meta', 'artist_name_live']

In [49]:
df_final[[
    "chartmetric_id",
    "requested_artist_name",
    "artist_name_meta",
    "artist_name_live"
]].head(15)

,chartmetric_id,requested_artist_name,artist_name_meta,artist_name_live
0,1103173,Murilo Huff,Murilo Huff,Murilo Huff
1,776,Queens of the Stone Age,Queens of the Stone Age,Queens of the Stone Age
2,113,Whitesnake,Whitesnake,NaN
3,7622,Espinoza Paz,Espinoza Paz,Espinoza Paz
4,566366,La Joaqui,La Joaqui,La Joaqui
5,4938,Red Velvet,Red Velvet,Red Velvet
6,213227,Los Invasores De Nuevo León,Los Invasores De Nuevo León,Los Invasores De Nuevo León
7,3630,Popcaan,Popcaan,Popcaan
8,675,Steve Miller Band,Steve Miller Band,Steve Miller Band
9,206351,Banda Los Recoditos,Banda Los Recoditos,Banda Los Recoditos


In [50]:
df_final["validacion_requested_vs_meta"] = (
    df_final["requested_artist_name"] == df_final["artist_name_meta"]
)

df_final["validacion_meta_vs_live"] = (
    df_final["artist_name_live"].isna() |
    (df_final["artist_name_meta"] == df_final["artist_name_live"])
)

In [51]:
print("Filas metadata:", df_metadata.shape[0])
 #print("Filas metadata sin 34:", df_metadata_sin34.shape[0])

print("Filas final:", df_final.shape[0])

Filas metadata: 4482
Filas final: 4482


## sin info de live events

In [52]:
print("Artistas sin info de live events:", df_final["artist_name_live"].isna().sum())

Artistas sin info de live events: 1970


In [53]:
df_final.loc[df_final["artist_name_live"].isna(), ["chartmetric_id", "artist_name_meta"]].head(10)

,chartmetric_id,artist_name_meta
2,113,Whitesnake
13,1423874,Guilherme & Benuto
16,50,Savage Garden
17,168422,John Martin
18,434307,Joe Jonas
19,814,The Hollies
21,18975,D.O.D
24,29975,Bomfunk MC's
27,1714066,Majed
28,145086,Alex Rose


In [54]:
print("Duplicados en df_final por chartmetric_id:", df_final["chartmetric_id"].duplicated().sum())

Duplicados en df_final por chartmetric_id: 0


In [56]:
print("Filas metadata:", df_metadata.shape[0])

print("Filas final:", df_final.shape[0])
print("Duplicados en df_final por chartmetric_id:", df_final["chartmetric_id"].duplicated().sum())
print("Artistas sin info de live events:", df_final["artist_name_live"].isna().sum())

Filas metadata: 4482
Filas final: 4482
Duplicados en df_final por chartmetric_id: 0
Artistas sin info de live events: 1970


# DF final

In [57]:
df_final.shape

(4482, 66)

In [58]:
df_final["validacion_meta_vs_live"].value_counts()

validacion_meta_vs_live
True     4481
False       1
Name: count, dtype: int64

In [60]:
df_final.loc[
    df_final["validacion_meta_vs_live"] == False,
    ["chartmetric_id", "artist_name_meta", "artist_name_live"]
]

,chartmetric_id,artist_name_meta,artist_name_live
4261,80180,Corrosion Of Conformity\t,Corrosion Of Conformity


In [61]:
df_final.columns

Index(['downloaded_at_utc', 'requested_chartmetric_id',
       'requested_artist_name', 'chartmetric_id', 'artist_name_meta', 'code2',
       'pronoun_title', 'gender_title', 'record_label', 'hometown_city',
       'cm_artist_rank', 'cm_artist_score', 'career_stage',
       'career_stage_score', 'career_trend', 'career_trend_score',
       'primary_genre', 'sp_followers', 'sp_monthly_listeners',
       'sp_popularity', 'sp_playlist_total_reach', 'ins_followers',
       'twitter_followers', 'tiktok_followers', 'tiktok_likes',
       'ycs_subscribers', 'ycs_views', 'youtube_daily_video_views',
       'youtube_monthly_video_views', 'deezer_fans', 'shazam_count',
       'pandora_lifetime_streams', 'pandora_lifetime_stations_added', 'band',
       'artist_name_live', 'n_shows_2024', 'avg_low_price_2024',
       'avg_high_price_2024', 'avg_venue_capacity_2024', 'total_capacity_2024',
       'n_shows_with_capacity_2024', 'n_cities_2024', 'n_countries_2024',
       'n_currencies_2024', 'main_c

## elimino columnas innecesarias

In [62]:
cols_drop = [
    "downloaded_at_utc",
    "requested_chartmetric_id",
    "requested_artist_name",
    "artist_name_live",
    "validacion_requested_vs_meta",
    "validacion_meta_vs_live"
]

df_final = df_final.drop(columns=cols_drop)

In [63]:
df_final = df_final.rename(columns={
    "artist_name_meta": "artist_name",
    "code2": "country"
})

In [64]:
print(df_final.shape)
df_final.head()

(4482, 60)


,chartmetric_id,artist_name,country,pronoun_title,gender_title,record_label,hometown_city,cm_artist_rank,cm_artist_score,career_stage,...,main_currency_2025,shows_per_country_2025,shows_per_country_2024,n_shows_total,n_shows_with_capacity_total,total_capacity_total,avg_venue_capacity_total,n_cities_total,n_countries_total,shows_per_country_total
0,1103173,Murilo Huff,BR,he/him,male,M Show,Goiania,1498,90.839004,superstar,...,NaN,2.000000,1.000000,3.0,2.0,45210.0,22605.000000,2.0,2.0,1.500000
1,776,Queens of the Stone Age,US,they/them,male,Matador,NaN,1620,90.569735,superstar,...,USD,2.411765,3.166667,60.0,40.0,312054.0,7801.350000,46.0,19.0,3.157895
2,113,Whitesnake,GB,they/them,NaN,WMG,London,1626,90.556828,legendary,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7622,Espinoza Paz,MX,he/him,male,Espinoza Paz,Mazatlán,1901,89.994779,superstar,...,USD,5.000000,1.000000,6.0,3.0,27424.0,9141.333333,3.0,1.0,6.000000
4,566366,La Joaqui,AR,she/her,NaN,LA JOAQUI / DALE PLAY Records,NaN,2354,89.171415,superstar,...,MXN,1.000000,NaN,2.0,1.0,1100.0,1100.000000,2.0,2.0,1.000000


In [65]:
df_final.shape

(4482, 60)

In [66]:
df_final.columns

Index(['chartmetric_id', 'artist_name', 'country', 'pronoun_title',
       'gender_title', 'record_label', 'hometown_city', 'cm_artist_rank',
       'cm_artist_score', 'career_stage', 'career_stage_score', 'career_trend',
       'career_trend_score', 'primary_genre', 'sp_followers',
       'sp_monthly_listeners', 'sp_popularity', 'sp_playlist_total_reach',
       'ins_followers', 'twitter_followers', 'tiktok_followers',
       'tiktok_likes', 'ycs_subscribers', 'ycs_views',
       'youtube_daily_video_views', 'youtube_monthly_video_views',
       'deezer_fans', 'shazam_count', 'pandora_lifetime_streams',
       'pandora_lifetime_stations_added', 'band', 'n_shows_2024',
       'avg_low_price_2024', 'avg_high_price_2024', 'avg_venue_capacity_2024',
       'total_capacity_2024', 'n_shows_with_capacity_2024', 'n_cities_2024',
       'n_countries_2024', 'n_currencies_2024', 'main_currency_2024',
       'n_shows_2025', 'avg_low_price_2025', 'avg_high_price_2025',
       'avg_venue_capacity_2

# SAVE  cohorte 2 
(hasta 1500 28 de marzo)
(hasta 3003 31/3/26)
(hasta 4482 3/4/26)

In [67]:
#df_final.to_csv("cohorte2_raw_1500.csv", index=False) #28-3-26

# df_final.to_csv("cohorte2_raw_3003.csv", index=False) #31-3-26

df_final.to_csv("cohorte2_raw_4482.csv", index=False) #3-4-26

